<a href="https://colab.research.google.com/github/Sahana-R-123/Fashion_MNIST/blob/main/Question2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
#Question 2
import wandb
from keras.datasets import fashion_mnist
import numpy as np

In [89]:
wandb.init(project='Fashion-MNIST',name='feed-forwardnn')
(x_train,y_train),(x_test,y_test)=fashion_mnist.load_data()

In [90]:
#preprocess
np.random.seed(42)
x_train = x_train / 255.0
x_test = x_test / 255.0
x_train = x_train.reshape(x_train.shape[0], 784)
x_test = x_test.reshape(x_test.shape[0], 784)

In [91]:
#one hot encode labels
def one_hot_encode(y, num_classes=10):
    return np.eye(num_classes)[y]

y_train_onehot = one_hot_encode(y_train)
y_test_onehot = one_hot_encode(y_test)


In [92]:
#define class
class feedforwardnn:
  def __init__(self, input_size, hidden_layers, output_size, learning_rate):
    self.learning_rate = learning_rate
    self.layers= [input_size] + hidden_layers + [output_size]
    self.weights = [np.random.randn(self.layers[i], self.layers[i + 1])  * np.sqrt(1/self.layers[i]) for i in range(len(self.layers) - 1)]
    self.biases = [np.zeros((1, self.layers[i + 1])) for i in range(len(self.layers) - 1)]

  #activation fn
  def sigmoid(self,x):
    return 1/(1+np.exp(-x))
  def relu(self, x):
    return np.maximum(0, x)

  #derivative
  def sigmoid_der(self,x):
    return x*(1-x)
  def relu_der(self, x):
      return (x > 0).astype(float)
  #output layer
  def softmax(self,x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x/np.sum(e_x,axis=1,keepdims=True)

  #forward propogation
  def forward_prop(self,x):
    activations=[x]
    for i in range(len(self.weights)-1):
      z=np.dot(activations[i],self.weights[i])+self.biases[i]
      #a=self.sigmoid(z)
      a=self.relu(z)
      activations.append(a)
    z= np.dot(activations[-1], self.weights[-1]) + self.biases[-1] #final weights and bias
    final_output=self.softmax(z)
    activations.append(final_output)
    return activations

  #compute loss
  def compute_loss(self,y_true, y_hat, batch_size):

    l = (-1 * np.sum(np.multiply(y_true, np.log(y_hat + 1e-8)))) / batch_size
    return l

  #backward propogation
  def backward_prop(self, x, y_true, activations):
    m=x.shape[0]
    deltas = [activations[-1] - y_true]
    for i in range(len(self.weights)-2, -1, -1):
      delta = np.dot(deltas[0],self.weights[i+1].T)
      #delta = delta * self.sigmoid_der(activations[i+1])
      delta = delta * self.relu_der(activations[i+1])
      deltas.insert(0, delta)
    dW = []
    dB = []
    for i in range(len(self.weights)):
      dw = np.dot(activations[i].T,deltas[i]) / m
      db = np.sum(deltas[i],axis=0,keepdims=True) / m

      dW.append(dw)
      dB.append(db)
    return dW,dB

  #update weights
  def update_para(self, dw, db):
    for i in range(len(self.weights)):
      self.weights[i] -= (self.learning_rate * dw[i])
      self.biases[i] -= (self.learning_rate * db[i])

  #calculate accuracy
  def calculate_accuracy(self,y_true, y_pred):
      return np.mean(np.argmax(y_true, axis=1) == np.argmax(y_pred, axis=1))

  #Gradient descent
  def gradient_descent(self, x, y_true, epochs, batch_size):
      for epoch in range(epochs):
        activations = self.forward_prop(x)
        dw, db = self.backward_prop(x, y_true, activations)
        self.update_para(dw, db)
        final_output = activations[-1]
        loss = self.compute_loss(y_true,final_output, x.shape[0])
        accuracy = self.calculate_accuracy(y_true, final_output)
        print(f"Epoch: {epoch}, Loss: {loss}, Accuracy: {accuracy}")
        wandb.log({"epoch": epoch,"loss": loss,"accuracy": accuracy})

   #Prediction
  def predict(self, X):

          predictions = np.argmax(self.forward_prop(X)[-1],axis=1)
          return predictions

In [93]:
#hyperparameters
input_size = 784
hidden_layers = [128, 64]
output_size = 10
learning_rate = 0.1
epochs = 50
batch_size = 128

#create model
model = feedforwardnn(input_size,hidden_layers,output_size,learning_rate)

# train model
model.gradient_descent(x_train,y_train_onehot,epochs,batch_size)

# test accuracy
test_output = model.forward_prop(x_test)[-1]
test_accuracy = model.calculate_accuracy(y_test_onehot,test_output)

print(f"Test Accuracy : {test_accuracy:.4f}")
wandb.log({"test_accuracy": test_accuracy})
wandb.finish()

Epoch: 0, Loss: 2.3206530135173877, Accuracy: 0.1179
Epoch: 1, Loss: 2.2360928452396287, Accuracy: 0.16648333333333334
Epoch: 2, Loss: 2.1749735941051935, Accuracy: 0.19596666666666668
Epoch: 3, Loss: 2.1199333042037765, Accuracy: 0.23631666666666667
Epoch: 4, Loss: 2.06365626801121, Accuracy: 0.2901166666666667
Epoch: 5, Loss: 2.006882188779161, Accuracy: 0.3792833333333333
Epoch: 6, Loss: 1.9489337185649185, Accuracy: 0.45248333333333335
Epoch: 7, Loss: 1.8900620141868858, Accuracy: 0.4984
Epoch: 8, Loss: 1.830112499276838, Accuracy: 0.53885
Epoch: 9, Loss: 1.7686497380023085, Accuracy: 0.5779166666666666
Epoch: 10, Loss: 1.7061026793452558, Accuracy: 0.6095333333333334
Epoch: 11, Loss: 1.6444601954743132, Accuracy: 0.62705
Epoch: 12, Loss: 1.5842616840084411, Accuracy: 0.6376166666666667
Epoch: 13, Loss: 1.5263409405798152, Accuracy: 0.6434166666666666
Epoch: 14, Loss: 1.4716095240085703, Accuracy: 0.6498
Epoch: 15, Loss: 1.4202167820195875, Accuracy: 0.65265
Epoch: 16, Loss: 1.3722

accuracy,▁▂▂▂▅▆▇▇▇▇██████████████████████████████
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
loss,██▇▇▇▆▆▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
test_accuracy,▁
accuracy,0.68375
epoch,49
loss,0.87921
test_accuracy,0.6788
